# Use Your Trained Mini GPT-2 Model

This notebook loads and uses the trained model you exported.

**Make sure these files are in the same folder:**
- `mini_gpt_model.pt` (full model with architecture + weights)
- `mini_gpt_vocab.json` (vocabulary)

## 1. Import libraries and load model

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import json

# Load the full model (architecture + weights)
model = torch.load('mini_gpt_model.pt')
model.eval()

# Load vocabulary
with open('mini_gpt_vocab.json', 'r') as f:
    vocab_data = json.load(f)

stoi = vocab_data['stoi']
itos = {int(k): v for k, v in vocab_data['itos'].items()}
vocab_size = vocab_data['vocab_size']
block_size = vocab_data['block_size']

# Determine device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Model loaded successfully")
print(f"  Total parameters: {total_params:,}")
print(f"  Vocab size: {vocab_size}")
print(f"  Block size: {block_size}")
print(f"  Device: {DEVICE}")

## 2. Encoding, decoding, and text generation

In [ ]:
# Encode and decode functions
def encode(s):
    return [stoi[c] for c in s]

def decode(indices):
    return ''.join([itos[i] for i in indices])

def generate_text(prompt, strategy='temperature', max_tokens=200, temperature=1.0, top_k=None, top_p=0.9):
    """
    Generate text from the trained model.
    
    Args:
        prompt: Starting text (e.g., "Patient: ")
        strategy: 'greedy', 'temperature', 'top_k', or 'top_p'
        max_tokens: How many tokens to generate
        temperature: Scale for randomness (only for temperature strategy)
        top_k: Keep top k tokens (only for top_k strategy)
        top_p: Keep tokens summing to top p probability (only for top_p strategy)
    """
    encoded = torch.tensor([encode(prompt)], device=DEVICE, dtype=torch.long)
    
    if top_k is None:
        top_k = min(50, vocab_size)
    
    model.eval()
    with torch.no_grad():
        for _ in range(max_tokens):
            logits, _ = model(encoded[:, -block_size:])
            logits = logits[0, -1, :]
            
            if strategy == 'greedy':
                next_token = logits.argmax().unsqueeze(0).unsqueeze(0)
            
            elif strategy == 'temperature':
                logits = logits / temperature
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, 1).unsqueeze(0)
            
            elif strategy == 'top_k':
                top_k_vals, top_k_indices = torch.topk(logits, min(top_k, vocab_size))
                logits_filtered = torch.full_like(logits, float('-inf'))
                logits_filtered[top_k_indices] = top_k_vals
                probs = F.softmax(logits_filtered, dim=-1)
                next_token = torch.multinomial(probs, 1).unsqueeze(0)
            
            elif strategy == 'top_p':
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                sorted_probs = F.softmax(sorted_logits, dim=-1)
                cumsum_probs = torch.cumsum(sorted_probs, dim=-1)
                mask = cumsum_probs <= top_p
                mask[0] = True
                filtered_logits = torch.full_like(logits, float('-inf'))
                filtered_logits[sorted_indices[mask]] = sorted_logits[mask]
                probs = F.softmax(filtered_logits, dim=-1)
                next_token = torch.multinomial(probs, 1).unsqueeze(0)
            
            encoded = torch.cat([encoded, next_token], dim=1)
    
    return decode(encoded[0].tolist())

print("✓ Generation functions ready")

## 5. Generate samples

In [ ]:
# Example prompts
prompts = [
    "Patient: ",
    "Doctor: ",
    "Healthcare ",
    "The "
]

for prompt in prompts:
    if all(c in stoi for c in prompt):
        print(f"\nPrompt: '{prompt}'")
        print("-" * 60)
        result = generate_text(prompt, strategy='temperature', temperature=1.0, max_tokens=150)
        print(result)
    else:
        print(f"\nPrompt '{prompt}' contains unknown characters, skipping")

## 6. Try different sampling strategies on same prompt

In [ ]:
test_prompt = "Patient: I"

print(f"Prompt: '{test_prompt}'")
print("=" * 70)

print("\n1. GREEDY (deterministic):")
print(generate_text(test_prompt, strategy='greedy', max_tokens=100))

print("\n2. TEMPERATURE 0.5 (confident):")
print(generate_text(test_prompt, strategy='temperature', temperature=0.5, max_tokens=100))

print("\n3. TEMPERATURE 1.5 (creative):")
print(generate_text(test_prompt, strategy='temperature', temperature=1.5, max_tokens=100))

print("\n4. TOP-P 0.9 (nucleus sampling):")
print(generate_text(test_prompt, strategy='top_p', top_p=0.9, max_tokens=100))

## 7. Interactive generation - customise here

In [ ]:
# ===== CUSTOMIZE THESE =====
my_prompt = "Patient: I have a "  # Change this to your prompt
my_strategy = "temperature"      # Options: 'greedy', 'temperature', 'top_k', 'top_p'
my_temperature = 1.0             # Only used if strategy='temperature'
my_max_tokens = 200              # How long to generate
# ===========================

print(f"Prompt: '{my_prompt}'")
print("-" * 70)
result = generate_text(
    my_prompt,
    strategy=my_strategy,
    temperature=my_temperature,
    max_tokens=my_max_tokens
)
print(result)